# Conversational memory agent — interactive tour

A LangGraph ReAct agent (`langchain.agents.create_agent`) that answers from the
news corpus via a retriever tool, with its conversation state persisted in
AgensGraph by **`AgensSaver`** (the LangGraph checkpointer). The same `thread_id`
resumes the conversation — even from a brand-new agent instance. The transcript
is also mirrored to **`AgensChatMessageHistory`**.

**Prerequisites:** the news store (demo 03's `ingest.py`) and an `OPENAI_API_KEY`.

In [1]:
import sys, pathlib

HERE = pathlib.Path.cwd()                  # .../04_chat_memory_agent
p = HERE
while p != p.parent and not (p / "_common").is_dir():
    p = p.parent
sys.path.insert(0, str(p))                 # demos root, for _common
sys.path.insert(0, str(HERE))              # this dir, for `agent`

import agent                               # build_agent / ask / search_news tool live here
from _common import agens
from langchain_core.messages import AIMessage, HumanMessage
from langchain_agensgraph import AgensChatMessageHistory

THREAD = "notebook-demo"
bot, saver = agent.build_agent()           # ReAct agent + AgensSaver checkpointer
saver.delete_thread(THREAD)                # start this demo thread clean
print("agent ready — AgensSaver checkpointing graph 'agent_memory'")

agent ready — AgensSaver checkpointing graph 'agent_memory'


## A multi-turn conversation

Each turn is persisted by the checkpointer under `THREAD`; later turns see the
earlier ones (and the tool results) without re-sending them.

In [2]:
print(agent.ask(bot, THREAD,
    "Search the news for stories about artificial intelligence and summarize the main themes."))

The search for recent news articles about artificial intelligence (AI) yielded several key themes and discussions:

1. **Job Automation and Augmentation**: There is ongoing concern about AI's impact on the job market, particularly regarding automation. Discussions focus on how AI can augment human capabilities rather than completely replace jobs. This includes exploring ways to safeguard workers from job loss due to automation.

2. **Ethical Considerations**: The ethical implications of AI development are a significant topic. There is a consensus that ensuring AI is developed and used ethically is a collective responsibility, emphasizing the need for public engagement and awareness as technology advances rapidly.

3. **Applications of AI**: Various applications of AI are highlighted, including its use in customer service, hiring processes, and even in education. Companies are increasingly leveraging AI to enhance efficiency and decision-making.

4. **Future of AI**: The evolution of AI

In [3]:
print(agent.ask(bot, THREAD, "Which of those themes relates most to jobs or hiring?"))

The theme that relates most to jobs or hiring is **Job Automation and Augmentation**. This theme encompasses the following aspects:

- **Impact on Employment**: There is significant concern about how AI and automation may lead to job displacement, as machines and algorithms take over tasks traditionally performed by humans.

- **Augmentation vs. Replacement**: The discussion includes the potential for AI to augment human capabilities rather than completely replace jobs. This perspective emphasizes the importance of finding ways to integrate AI into the workforce in a manner that enhances productivity while preserving employment opportunities.

- **Hiring Processes**: Companies are increasingly using AI in hiring and human resources, leveraging machine learning algorithms to streamline recruitment, assess candidates, and make data-driven decisions.

Overall, this theme highlights the dual nature of AI's impact on the job market, where it can both pose challenges and offer opportunities 

In [4]:
print(agent.ask(bot, THREAD, "Give one concrete example from the articles you found."))

One concrete example related to jobs and hiring from the articles is the use of AI in the recruitment process. Companies are leveraging AI technologies to enhance their hiring practices by utilizing machine learning algorithms to analyze resumes, assess candidate qualifications, and even predict job performance.

For instance, AI tools can automate the initial screening of applications, allowing HR teams to focus on more strategic aspects of recruitment. This not only speeds up the hiring process but also helps in identifying the best candidates based on data-driven insights rather than solely relying on human judgment.

This application of AI in hiring illustrates how technology can augment human resources functions, making them more efficient while also raising discussions about the ethical implications and potential biases in AI-driven decision-making.


## Resume from a checkpoint — a brand-new agent instance

`build_agent()` constructs a fresh agent + a fresh `AgensSaver` (as a new process
would). Using the same `thread_id`, it picks up the full prior state from
AgensGraph — so it can answer about earlier turns without searching again.

In [5]:
bot2, _ = agent.build_agent()             # fresh instance, same persisted thread
print(agent.ask(bot2, THREAD,
    "Without searching again, what was my very first question in this conversation?"))

Your very first question was to search the news for stories about artificial intelligence and summarize the main themes.


## `AgensChatMessageHistory` — a simple per-session message log

A lighter-weight memory primitive: append/read messages keyed by a session id.

In [6]:
history = AgensChatMessageHistory(
    THREAD, graph=agens.make_graph("chat_log", create=True, refresh_schema=False))
history.clear()
history.add_messages([
    HumanMessage(content="What are the main AI themes in the news?"),
    AIMessage(content="Applications, job automation/augmentation, corporate adoption, AGI concerns."),
])
print(f"{len(history.messages)} messages for session {THREAD!r}:")
for m in history.messages:
    print(f"  {m.type:9} {m.content[:60]}")

2 messages for session 'notebook-demo':
  human     What are the main AI themes in the news?
  ai        Applications, job automation/augmentation, corporate adoptio


## What you can do with this

- **Durable agents**: `AgensSaver` persists full LangGraph state per `thread_id`,
  so conversations survive process restarts and resume exactly where they left off.
- **Grounded tools**: the agent answers from the AgensgraphVector news store
  (demo 03) via a retriever tool — graph, vectors, and agent memory in one DB.
- **Chat history**: `AgensChatMessageHistory` is a drop-in per-session message log.

Run one turn at a time from the shell to see true cross-process resume:

```bash
.venv/bin/python examples/demos/04_chat_memory_agent/agent.py my-thread "your message"
```

Close the shared pool when done: `agens.close()`